In [17]:
import os, glob
import numpy as np
import tensorflow as tf
import pandas as pd

## Prepare dataset

In [18]:
CAPTURES = "../dataset/captures"
MAX_LEN  = 100 
N_CH = 6 
classes = sorted(d for d in os.listdir(CAPTURES)
                 if os.path.isdir(os.path.join(CAPTURES, d)))

class_to_id = {c: i for i, c in enumerate(classes)}

lengths = []
"""
Load and preprocess the gesture recordings to create the dataset for training the MLP.

For each gesture class, the function:
1. Loads the accelerometer and gyroscope measurements from CSV files.
2. Creates a fixed-size sequence using zero-padding or truncation.
3. Flattens the 2D sensor matrix into a 1D feature vector.
4. Assigns a numerical label to identify the gesture.

Finally, X contains the input features and y contains the corresponding gesture labels.
"""

X, y = [], []
for cls in classes:
    for path in glob.glob(os.path.join(CAPTURES, cls, "*.csv")):
        rows = np.loadtxt(path, delimiter=",", skiprows=1, usecols=range(4, 10))
        seq = np.zeros((MAX_LEN, N_CH), dtype=np.float32)   
        n = min(rows.shape[0], MAX_LEN)
        seq[:n] = rows[:n]
        X.append(seq.flatten())          
        y.append(class_to_id[cls])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print("X", X.shape, "classes", len(classes))  


X (492, 600) classes 16


## Analyze dataset

In [15]:
print(X.shape)

(492, 600)


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)   # stratify: 16 classes

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)


model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(N_CH * MAX_LEN,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64,  activation="relu"),
    tf.keras.layers.Dense(len(classes), activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(X_train, y_train, batch_size=32, epochs=100, validation_data=(X_test, y_test))

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.4300 - loss: 1.9989 - val_accuracy: 0.6364 - val_loss: 1.2420
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7176 - loss: 0.9471 - val_accuracy: 0.7980 - val_loss: 0.7716
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8524 - loss: 0.6027 - val_accuracy: 0.9091 - val_loss: 0.5295
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9008 - loss: 0.4023 - val_accuracy: 0.8990 - val_loss: 0.3977
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9211 - loss: 0.2933 - val_accuracy: 0.9192 - val_loss: 0.3340
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9415 - loss: 0.2337 - val_accuracy: 0.9293 - val_loss: 0.3076
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9440 - loss: 0.1942 - val_accuracy: 0.9192 - val_loss: 0.2954
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9491 - loss: 0.1671 - val_accuracy: 0.9192 - 